### 1. Environment Setup

#### A. Importing Libraries

In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn


print(f"Pandas version: {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")

Pandas version: 3.0.5
scikit-learn version: 1.9.0


#### B. Display Settings

In [2]:
# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

### 2. Data Loading and Schema Overview

#### A. Load Datasets

In [3]:
df = pd.read_csv(
    'olist_cleansed.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96470 entries, 0 to 96469
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96470 non-null  str           
 1   customer_id                    96470 non-null  str           
 2   order_status                   96470 non-null  str           
 3   order_purchase_timestamp       96470 non-null  datetime64[us]
 4   order_approved_at              96456 non-null  datetime64[us]
 5   order_delivered_carrier_date   96469 non-null  datetime64[us]
 6   order_delivered_customer_date  96470 non-null  datetime64[us]
 7   order_estimated_delivery_date  96470 non-null  datetime64[us]
 8   actual_delivery_days           96470 non-null  float64       
 9   estimated_delivery_days        96470 non-null  float64       
 10  is_late                        96470 non-null  int64         
 11  item_count                

#### B. Importing Necessary Files for Distance Feature

In [4]:
# load datasets required for seller-to-customer distance
order_items = pd.read_csv('./data/olist_order_items_dataset.csv')
seller = pd.read_csv('./data/olist_sellers_dataset.csv')
geolocation = pd.read_csv('./data/olist_geolocation_dataset.csv')

In [5]:
print(f'Order items: {order_items.shape}')
print(f'Sellers: {seller.shape}')
print(f'Geolocation: {geolocation.shape}')

Order items: (112650, 7)
Sellers: (3095, 4)
Geolocation: (1000163, 5)


#### C. Prepare Geolocation Data

In [6]:
# remove coordinates outside Brazil's approximate geographic bounds
valid_coords = (
    (geolocation['geolocation_lat'] >= -34.0) & (geolocation['geolocation_lat'] <= 5.5) &
    (geolocation['geolocation_lng'] >= -74.0) & (geolocation['geolocation_lng'] <= -34.0)
)

geolocation_clean = geolocation[valid_coords].copy()

# create one representative coordinate per ZIP-code prefix
geo_zip_lookup = geolocation_clean.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].median().reset_index()

print(f"=== Clean Geolocation Lookup Table ===")
print(f"Original rows: {len(geolocation):,}")
print(f"Cleaned unique ZIP codes: {len(geo_zip_lookup):,}")

geo_zip_lookup.head()

=== Clean Geolocation Lookup Table ===
Original rows: 1,000,163
Cleaned unique ZIP codes: 19,010


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,1001,-23.55,-46.63
1,1002,-23.55,-46.64
2,1003,-23.55,-46.64
3,1004,-23.55,-46.63
4,1005,-23.55,-46.64


#### D. Identify Distinct Seller For Each Order

In [7]:
order_sellers = (
    order_items[['order_id', 'seller_id']].drop_duplicates().merge(
        seller[['seller_id', 'seller_zip_code_prefix']],
        on='seller_id',
        how='left'
    )
)

print(f'Unique order-seller pairs: {len(order_sellers)}')
print(f'Missing seller ZIP codes: {order_sellers['seller_zip_code_prefix'].isna().sum()}')

order_sellers.head()

Unique order-seller pairs: 100010
Missing seller ZIP codes: 0


,order_id,seller_id,seller_zip_code_prefix
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900


#### E. Joining Geolocation Information with Order_Seller

In [8]:
# prepare seller coordinate lookup
seller_geo = geo_zip_lookup.rename(columns={
    'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
    'geolocation_lat': 'seller_lat',
    'geolocation_lng': 'seller_lng'
})

# add seller coordinates
order_sellers = order_sellers.merge(
    seller_geo,
    on='seller_zip_code_prefix',
    how='left'
)

print(f'Order-seller pairs: {len(order_sellers)}')
print(f'Missing seller coordinates: {order_sellers['seller_lat'].isna().sum()}')

order_sellers.head()

Order-seller pairs: 100010
Missing seller coordinates: 220


,order_id,seller_id,seller_zip_code_prefix,seller_lat,seller_lng
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277,-22.50,-44.13
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471,-23.56,-46.52
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564,-22.27,-46.17
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403,-20.55,-47.39
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900,-22.93,-53.14


#### F. Customer Side Geolocation Information

In [9]:
# prepare customer coordinate lookup
customer_geo = geo_zip_lookup.rename(columns={
    'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
    'geolocation_lat': 'customer_lat',
    'geolocation_lng': 'customer_lng'
})

# add customer ZIP code to each order-seller pair
order_sellers = order_sellers.merge(
    df[['order_id', 'customer_zip_code_prefix']],
    on='order_id',
    how='inner'
)

# add customer coordinates
order_sellers = order_sellers.merge(
    customer_geo,
    on='customer_zip_code_prefix',
    how='left'
)

print(f'Order-sellers pairs in modelling dataset: {len(order_sellers)}')
print(f'Missing seller coordinates: {order_sellers['seller_lat'].isna().sum()}')
print(f'Missing customer coordinates: {order_sellers['customer_lat'].isna().sum()}')

order_sellers.head()

Order-sellers pairs in modelling dataset: 97811
Missing seller coordinates: 217
Missing customer coordinates: 271


,order_id,seller_id,seller_zip_code_prefix,seller_lat,seller_lng,customer_zip_code_prefix,customer_lat,customer_lng
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277,-22.50,-44.13,28013,-21.76,-41.31
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471,-23.56,-46.52,15775,-20.21,-50.93
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564,-22.27,-46.17,35661,-19.87,-44.59
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403,-20.55,-47.39,12952,-23.10,-46.60
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900,-22.93,-53.14,13226,-23.25,-46.83


#### G. Calculate Straight-Line Seller-to-Customer Distance (Haversine Formula)

Haversine: Standard formula for calculating the approximate distance between two latitude-longitude coordinates while accounting for the curvature of the Earth.

Limitation: Haversine gives geographic straight-line distance, not actual road or delivery route distance, so it is an approximate indicator of delivery distance

In [10]:
# Calculate straight-line seller-to-customer distance using Haversine formula
lat1 = np.radians(order_sellers['seller_lat'])
lon1 = np.radians(order_sellers['seller_lng'])
lat2 = np.radians(order_sellers['customer_lat'])
lon2 = np.radians(order_sellers['customer_lng'])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

order_sellers['seller_customer_distance_km'] = (
    6371 * 2 * np.arcsin(np.sqrt(a))
)

order_sellers[
    ['order_id', 'seller_id', 'seller_customer_distance_km']
].head()

,order_id,seller_id,seller_customer_distance_km
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,301.23
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,588.40
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,312.84
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,295.03
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,646.47


#### H. Validate Seller Structure and Missing Distances

In [11]:
seller_counts = order_sellers.groupby('order_id')['seller_id'].nunique()

print(f'Single-seller orders: {(seller_counts == 1).sum()}')
print(f'Multi-seller orders: {(seller_counts > 1).sum()}')
print(f'Maximum sellers in one order: { seller_counts.max()}')

Single-seller orders: 95195
Multi-seller orders: 1275
Maximum sellers in one order: 5


In [12]:
# records with missing distance
missing_distance = order_sellers[order_sellers['seller_customer_distance_km'].isna()]

print(f'Missing order-seller distances: {len(missing_distance)}')
print(f'Orders affected: {missing_distance['order_id'].nunique()}')

# for orders with missing sellers, the split single-seller vs multi-seller?
affected_orders = seller_counts.loc[seller_counts.index.isin(missing_distance['order_id'])]

print(f'Affected single-seller orders: {(affected_orders == 1).sum()}')
print(f'Affected multi-seller orders: {(affected_orders > 1).sum()}')

Missing order-seller distances: 487
Orders affected: 481
Affected single-seller orders: 471
Affected multi-seller orders: 10


#### I. Aggregate Seller-to-Customer Distance to Order Level

In [13]:
# Average seller-to-customer distance per order
# Require all sellers in the order to have a valid distance
order_distance = (
    order_sellers
    .groupby('order_id')['seller_customer_distance_km']
    # for each order_id, if distance exists for all sellers, compute the mean distance, else nan
    .agg(lambda x: x.mean() if x.notna().all() else np.nan)
    .reset_index() # turns order_id back into a normal column
    .rename(columns={
        'seller_customer_distance_km': 'avg_seller_customer_distance_km'
    })
)

print('Orders:', len(order_distance))
print(
    'Orders with missing distance:',
    order_distance['avg_seller_customer_distance_km'].isna().sum()
)

order_distance['avg_seller_customer_distance_km'].describe()

Orders: 96470
Orders with missing distance: 481


count   95989.00
mean      600.46
std       592.15
min         0.00
25%       186.87
50%       433.90
75%       797.34
max      3398.49
Name: avg_seller_customer_distance_km, dtype: float64

#### J. Add Distance Feature to Modelling Dataset

In [14]:
df = df.merge(
    order_distance,
    on='order_id',
    how='left'
)

print(df.shape)
print(f'Missing distances: {df['avg_seller_customer_distance_km'].isna().sum()}')

(96470, 24)
Missing distances: 481


### 3. Splitting of Datasets

#### A. Train / Validation / Test Split

The dataset is split chronologically using `order_purchase_timestamp`, with 70% of
orders used for training, 15% for validation and 15% for testing.

A chronological split is used because the intended prediction setting is operational:
the model will use historical orders to make predictions for future orders at the time
of purchase. Keeping later orders in the validation and test sets therefore provides
a more realistic evaluation than randomly mixing earlier and later orders.

The validation set will be used during model development and comparison, while the
test set will remain untouched until the final model evaluation.

Since the classification target is imbalanced, the proportion and number of late
orders are checked after the chronological split. The data are not stratified because
doing so could disrupt the temporal ordering. The naturally occurring class
distribution is retained in each period.

No random seed is required for the train/validation/test split because the data are
split deterministically according to chronological order rather than through random
sampling. A fixed random seed will be set and documented for any later modelling or
resampling procedures that involve randomness, to ensure reproducibility.

In [15]:
# sort orders chronologically by purchase timestamp
df_model = (
    df
    .sort_values('order_purchase_timestamp')
    .reset_index(drop=True)
)

# define 70/15/15 split points
n = len(df_model)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

# create chronological train, validation and test sets
train_df = df_model.iloc[:train_end].copy()
validation_df = df_model.iloc[train_end:validation_end].copy()
test_df = df_model.iloc[validation_end:].copy()

In [16]:
# check date range and class distribution of each split
# because split is based on chronological order, the percentage of late orders is not expected
# to be identical across the three split datasets
def describe_split(name, data):
    print(f"{name}")
    print(f"Orders: {len(data):,}")
    print(
        # check that chronological order is correct
        f"Date range: "
        f"{data['order_purchase_timestamp'].min().date()} "
        f"to {data['order_purchase_timestamp'].max().date()}"
    )

    # check that each set still contains enough late orders for classification evaluation
    class_counts = data['is_late'].value_counts().sort_index()
    class_pct = data['is_late'].value_counts(
        normalize=True
    ).sort_index() * 100

    print(
        f"On time: {class_counts.get(0, 0):,} "
        f"({class_pct.get(0, 0):.2f}%)"
    )
    print(
        f"Late:    {class_counts.get(1, 0):,} "
        f"({class_pct.get(1, 0):.2f}%)"
    )
    print()


describe_split("TRAIN", train_df)
describe_split("VALIDATION", validation_df)
describe_split("TEST", test_df)

TRAIN
Orders: 67,529
Date range: 2016-09-15 to 2018-04-15
On time: 62,239 (92.17%)
Late:    5,290 (7.83%)

VALIDATION
Orders: 14,470
Date range: 2018-04-15 to 2018-06-21
On time: 13,846 (95.69%)
Late:    624 (4.31%)

TEST
Orders: 14,471
Date range: 2018-06-21 to 2018-08-29
On time: 13,851 (95.72%)
Late:    620 (4.28%)



**Split check:**  
The chronological split produced 67,529 training orders, 14,470 validation orders
and 14,471 test orders. The late-delivery rate decreased from 7.83% in training to
4.31% in validation and 4.28% in testing. However, the validation and test sets still
contain 624 and 620 late orders respectively, providing sufficient minority-class
observations for evaluation.

The difference in class proportions is retained rather than artificially balanced,
as it reflects the naturally observed distribution across the different time periods.

In [17]:
# For Report 1
print("TRAIN")
print("Orders:", len(train_df))
print("Purchase dates:",
      train_df['order_purchase_timestamp'].min(),
      "to",
      train_df['order_purchase_timestamp'].max())
print("Median actual delivery days:",
      train_df['actual_delivery_days'].median())
print("Late delivery rate:",
      train_df['is_late'].mean())

print("\nVALIDATION")
print("Orders:", len(validation_df))
print("Purchase dates:",
      validation_df['order_purchase_timestamp'].min(),
      "to",
      validation_df['order_purchase_timestamp'].max())
print("Median actual delivery days:",
      validation_df['actual_delivery_days'].median())
print("Late delivery rate:",
      validation_df['is_late'].mean())

print("\nTEST")
print("Orders:", len(test_df))
print("Purchase dates:",
      test_df['order_purchase_timestamp'].min(),
      "to",
      test_df['order_purchase_timestamp'].max())
print("Median actual delivery days:",
      test_df['actual_delivery_days'].median())
print("Late delivery rate:",
      test_df['is_late'].mean())

TRAIN
Orders: 67529
Purchase dates: 2016-09-15 12:16:38 to 2018-04-15 20:12:35
Median actual delivery days: 11.475266203703704
Late delivery rate: 0.0783367145966918

VALIDATION
Orders: 14470
Purchase dates: 2018-04-15 20:17:11 to 2018-06-21 08:29:29
Median actual delivery days: 8.909386574074073
Late delivery rate: 0.04312370421561852

TEST
Orders: 14471
Purchase dates: 2018-06-21 08:41:07 to 2018-08-29 15:00:37
Median actual delivery days: 7.2290625
Late delivery rate: 0.04284430930827172


### 4. 